# 24.1 端侧 MLOps 与质量门禁

把导出/量化/设备指标收成可重复 gate。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
@dataclass
class GateResult:
    name: str
    ok: bool
    detail: dict


def gate_export(op_coverage: float, dynamic_shape_ok: bool) -> GateResult:
    ok = op_coverage >= 0.98 and dynamic_shape_ok
    return GateResult("export", ok, {"op_coverage": op_coverage, "dynamic_shape_ok": dynamic_shape_ok})


def gate_quant(ppl_delta: float, layer_cos_min: float) -> GateResult:
    ok = ppl_delta <= 0.5 and layer_cos_min >= 0.999
    return GateResult("quant", ok, {"ppl_delta": ppl_delta, "layer_cos_min": layer_cos_min})


def gate_device(ttft_ms: float, itl_ms: float, peak_mb: float, throttle_drop: float) -> GateResult:
    ok = ttft_ms <= 250 and itl_ms <= 45 and peak_mb <= 2800 and throttle_drop <= 0.25
    return GateResult("device", ok, {"ttft": ttft_ms, "itl": itl_ms, "peak_mb": peak_mb, "throttle_drop": throttle_drop})


def release_decision(gates: list[GateResult]) -> dict:
    failed = [g.name for g in gates if not g.ok]
    return {"release": len(failed)==0, "failed": failed, "gates": [g.__dict__ for g in gates]}


gates = [
    gate_export(0.991, True),
    gate_quant(0.35, 0.9994),
    gate_device(210, 38, 2300, 0.12),
]
print(json.dumps(release_decision(gates), ensure_ascii=False, indent=2))

# 金样本回归：简单字符串一致性/工具调用可解析率
GOLD = ["用一句话解释量化", "输出 JSON: {\"tool\":\"get_battery\",\"arguments\":{}}"]

def parse_ok(text: str) -> bool:
    try:
        obj = json.loads(text[text.index("{"):text.rindex("}")+1])
        return "tool" in obj
    except Exception:
        return False

simulated_outputs = ["量化是降低数值精度以省内存", "好的 {\"tool\":\"get_battery\",\"arguments\":{}}"]
print("gold pass rate", sum(True for o in simulated_outputs)/len(GOLD), "tool parse", parse_ok(simulated_outputs[1]))

## 小结

没有门禁的端侧发布等于盲飞；设备长稳与金样本要和签名/OTA 绑定。